In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from pathlib import Path


# ============================================================
# Configuration
# ============================================================

OUT_DIR = Path("vlm_visualization_outputs")
OUT_DIR.mkdir(exist_ok=True)

# Set this to your real CSV path.
# Expected columns:
# model, condition, question_class, accuracy
#
# condition should contain:
# "with video", "without videos"
#
# If CSV_PATH is None, the script generates fake demo data.
CSV_PATH = "/work/courses/3dv/team1/lmms-eval/diagonistic/vlm_visualization_outputs/vlm_results.csv"
# CSV_PATH = "your_vlm_results.csv"

VIDEO_COND = "with video"
NO_VIDEO_COND = "without videos"
CONDITIONS = [VIDEO_COND, NO_VIDEO_COND]

QUESTION_ORDER = [
    "Visibility",
    "Last visible",
    "Last placement",
    "Fixture",
    "Camera rel.",
    "Obj. relation",
    "Obj. distance",
]

QUESTION_RENAME = {
    "oos_step1_visibility": "Visibility",
    "oos_step2_last_visible": "Last visible",
    "oos_step3_last_placement": "Last placement",
    "oos_step4_fixture": "Fixture",
    "oos_branch_object_camera_relative_position": "Camera rel.",
    "oos_branch_object_object_relation": "Obj. relation",
    "oos_branch_object_object_distance": "Obj. distance",
}

# ============================================================
# Global display names
# Change names here only.
# Left side = internal name from data/script.
# Right side = display name in figure/table.
# ============================================================

DISPLAY_NAMES = {
    # Question names
    "Visibility": {
        "figure": "Visibility",
        "table": "Visibility",
    },
    "Last visible": {
        "figure": "Last Visible ",
        "table": "Last Vis",
    },
    "Last placement": {
        "figure": "Last Placement",
        "table": "Last Place",
    },
    "Fixture": {
        "figure": "Nearest Fixture",
        "table": "Nearest Fixture",
    },
    "Camera rel.": {
        "figure": "Obj-Cam Rel Pos",
        "table": "obj-Cam Rel Pos",
    },
    "Obj. relation": {
        "figure": "Obj-Obj Rel Pos",
        "table": "Obj-Obj Rel Pos",
    },
    "Obj. distance": {
        "figure": "Obj-Obj Distance",
        "table": "Obj-Obj Distance",
    },

    # Model names
    "Qwen3.6-27B": {
        "figure": "Qwen3.6-27B",
        "table": "Qwen3.6-27B",
    },
    "Qwen3-VL-235B-Instruct": {
        "figure": "Qwen3-VL-235B-Instruct",
        "table": "Qwen3-VL-235B-Instruct",
    },
    "InternVL3_5-8B": {
        "figure": "InternVL3_5-8B",
        "table": "InternVL3_5-8B",
    },
    "VLM-3r": {
        "figure": "VLM-3r",
        "table": "VLM-3r",
    },

    # Fixed table labels
    "Model": {
        "figure": "Model",
        "table": "Methods",
    },
    "Avg.": {
        "figure": "Avg.",
        "table": "Avg.",
    },
}


def display_name(name: str, target: str = "figure") -> str:
    """
    Return display name for figure/table.
    target should be either 'figure' or 'table'.
    If name is not found, returns the original name.
    """
    return DISPLAY_NAMES.get(name, {}).get(target, name)


BASELINE_LABEL = {
    "Visibility": "(chance 50%)",
    "Last visible": "",
    "Last placement": "",
    "Fixture": "(chance 20--33.33%)",
    "Camera rel.": "(chance 25%)",
    "Obj. distance": "(chance 33.33%)",
    "Obj. relation": "(chance 33.33%)",
}

CHANCE_BASELINE = {
    "Visibility": 0.50,
    "Last visible": None,
    "Last placement": None,
    "Fixture": None,          # variable: 20--33%
    "Camera rel.": 0.25,
    "Obj. distance": 1 / 3,
    "Obj. relation": 1 / 3,
}

# Optional grouping for table sections.
# Keep the keys as internal model names, not display names.
MODEL_GROUPS = {
    "Qwen3.6-27B": "General Models",
    "Qwen3-VL-235B-Instruct": "General Models",
    "InternVL3_5-8B": "General Models",
    "VLM-3r": "Specialized 3D Models",
}


# ============================================================
# Style
# ============================================================

plt.rcParams.update({
    "font.family": "DejaVu Serif",
    "font.size": 36,
    "axes.titlesize": 48,
    "axes.labelsize": 39,
    "xtick.labelsize": 33,
    "ytick.labelsize": 36,
    "legend.fontsize": 30,
    "figure.titlesize": 54,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


# ============================================================
# Data loading / fake data generation
# ============================================================

def generate_fake_data() -> pd.DataFrame:
    np.random.seed(17)

    models = [
        "Qwen3.6-27B",
        "Qwen3-VL-235B-Instruct",
        "InternVL3_5-8B",
        "VLM-3r",
    ]

    base_scores = {
        "Visibility": 0.82,
        "Last visible": 0.05,
        "Last placement": 0.08,
        "Fixture": 0.48,
        "Camera rel.": 0.28,
        "Obj. distance": 0.34,
        "Obj. relation": 0.25,
    }

    model_offsets = {
        "Qwen3.6-27B": 0.08,
        "Qwen3-VL-235B-Instruct": 0.06,
        "InternVL3_5-8B": 0.01,
        "VLM-3r": -0.04,
    }

    # Positive means "with video" tends to outperform "without videos".
    condition_effects = {
        "Visibility": -0.12,
        "Last visible": 0.02,
        "Last placement": 0.03,
        "Fixture": 0.12,
        "Camera rel.": 0.08,
        "Obj. distance": 0.00,
        "Obj. relation": 0.10,
    }

    rows = []

    for model in models:
        for q in QUESTION_ORDER:
            for condition in CONDITIONS:
                score = (
                    base_scores[q]
                    + model_offsets[model]
                    + (
                        condition_effects[q] / 2
                        if condition == VIDEO_COND
                        else -condition_effects[q] / 2
                    )
                    + np.random.normal(0, 0.03)
                )

                rows.append({
                    "model": model,
                    "question_class": q,
                    "condition": condition,
                    "accuracy": float(np.clip(score, 0, 1)),
                })

    return pd.DataFrame(rows)


def load_or_generate_data(csv_path=None) -> pd.DataFrame:
    if csv_path is None:
        df = generate_fake_data()
    else:
        df = pd.read_csv(csv_path)

    df = df.copy()

    required_cols = {"model", "condition", "question_class", "accuracy"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # Clean strings.
    df["model"] = df["model"].astype(str).str.strip()
    df["condition"] = df["condition"].astype(str).str.strip()
    df["question_class"] = df["question_class"].astype(str).str.strip()

    # Normalize raw question-class names if needed.
    df["question_class"] = df["question_class"].replace(QUESTION_RENAME)

    # Filter and validate.
    unknown_conditions = sorted(set(df["condition"]) - set(CONDITIONS))
    if unknown_conditions:
        print(f"Warning: dropping unknown conditions: {unknown_conditions}")

    unknown_questions = sorted(set(df["question_class"]) - set(QUESTION_ORDER))
    if unknown_questions:
        print(f"Warning: dropping unknown question classes: {unknown_questions}")

    df = df[df["condition"].isin(CONDITIONS)]
    df = df[df["question_class"].isin(QUESTION_ORDER)]

    if df.empty:
        raise ValueError(
            "No valid rows after filtering. Check that condition values are exactly "
            f"{CONDITIONS} and question classes match QUESTION_ORDER or QUESTION_RENAME."
        )

    df["accuracy"] = df["accuracy"].astype(float)

    # If there are duplicate rows, average them.
    df = (
        df.groupby(["model", "condition", "question_class"], as_index=False)["accuracy"]
        .mean()
    )

    return df


def get_score(df: pd.DataFrame, model: str, question_class: str, condition: str) -> float:
    subset = df[
        (df["model"] == model)
        & (df["question_class"] == question_class)
        & (df["condition"] == condition)
    ]

    if subset.empty:
        available = df[
            (df["model"] == model)
            & (df["question_class"] == question_class)
        ]["condition"].unique()

        raise ValueError(
            f"Missing result for model={model}, "
            f"question_class={question_class}, condition={condition}. "
            f"Available conditions for this model/question: {available}"
        )

    return float(subset["accuracy"].iloc[0])


# ============================================================
# Dot-line plot
# ============================================================

def plot_chance_aware_dotline(df: pd.DataFrame):
    model_order = (
        df.groupby("model")["accuracy"]
        .mean()
        .sort_values(ascending=False)
        .index
        .tolist()
    )

    color_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    model_colors = {
        model: color_cycle[i % len(color_cycle)]
        for i, model in enumerate(model_order)
    }

    fig, ax = plt.subplots(figsize=(13.5, 7.8))

    y_positions = {
        q: len(QUESTION_ORDER) - 1 - i
        for i, q in enumerate(QUESTION_ORDER)
    }

    offsets = np.linspace(-0.18, 0.18, len(model_order))

    # Fixture chance band.
    fixture_y = y_positions["Fixture"]
    ax.fill_betweenx(
        [fixture_y - 0.42, fixture_y + 0.42],
        0.20,
        1 / 3,
        color="#DCEAF3",
        alpha=0.9,
        zorder=0,
    )
    ax.text(
        0.265,
        fixture_y + 0.47,
        "fixture chance band",
        ha="center",
        va="bottom",
        fontsize=27,
        color="#333333",
    )

    # Fixed chance baselines.
    for q, chance in CHANCE_BASELINE.items():
        y = y_positions[q]
        if chance is not None:
            ax.vlines(
                chance,
                y - 0.43,
                y + 0.43,
                linestyles="--",
                linewidth=1.2,
                color="#555555",
                alpha=0.8,
                zorder=1,
            )

    # Horizontal guides.
    for q in QUESTION_ORDER:
        ax.axhline(
            y_positions[q],
            linewidth=0.6,
            color="#D0D0D0",
            alpha=0.8,
            zorder=0,
        )

    # Draw connected dots.
    model_handles = []

    for m_idx, model in enumerate(model_order):
        color = model_colors[model]
        y_offset = offsets[m_idx]

        for q in QUESTION_ORDER:
            y = y_positions[q] + y_offset

            with_video = get_score(df, model, q, VIDEO_COND)
            without_video = get_score(df, model, q, NO_VIDEO_COND)

            # Line from without videos to with video.
            ax.plot(
                [without_video, with_video],
                [y, y],
                color=color,
                linewidth=1.7,
                alpha=0.80,
                zorder=2,
            )

            # without videos: open circle.
            ax.scatter(
                without_video,
                y,
                s=64,
                facecolors="white",
                edgecolors=color,
                linewidths=1.7,
                zorder=3,
            )

            # with video: filled circle.
            ax.scatter(
                with_video,
                y,
                s=64,
                facecolors=color,
                edgecolors=color,
                linewidths=1.0,
                zorder=4,
            )

        model_handles.append(
            Line2D(
                [0],
                [0],
                color=color,
                marker="o",
                linestyle="-",
                label=display_name(model, "figure"),
                linewidth=1.7,
                markersize=7,
            )
        )

    # Axis labels.
    ytick_labels = [
        f"{display_name(q, 'figure')}\n{BASELINE_LABEL[q]}"
        for q in QUESTION_ORDER
    ]

    ax.set_yticks([y_positions[q] for q in QUESTION_ORDER])
    ax.set_yticklabels(ytick_labels)

    ax.set_xlim(0, 1.0)
    ax.set_xlabel("Accuracy")

    # Divider between groups.
    divider_y = (y_positions["Fixture"] + y_positions["Camera rel."]) / 2
    ax.axhline(divider_y, linewidth=1.0, color="#555555", alpha=0.85)

    ax.text(
        1.015,
        np.mean([y_positions["Visibility"], y_positions["Fixture"]]),
        "Incremental\n Step",
        ha="left",
        va="center",
        fontsize=33,
        fontweight="bold",
        transform=ax.transData,
    )

    ax.text(
        1.015,
        np.mean([y_positions["Camera rel."], y_positions["Obj. relation"]]),
        "Parallel\nBranches",
        ha="left",
        va="center",
        fontsize=33,
        fontweight="bold",
        transform=ax.transData,
    )

    # Legends.
    condition_handles = [
        Line2D(
            [0],
            [0],
            marker="o",
            color="#333333",
            markerfacecolor="white",
            linestyle="None",
            label=NO_VIDEO_COND,
            markersize=7,
        ),
        Line2D(
            [0],
            [0],
            marker="o",
            color="#333333",
            markerfacecolor="#333333",
            linestyle="None",
            label=VIDEO_COND,
            markersize=7,
        ),
        Line2D(
            [0],
            [0],
            color="#555555",
            linestyle="--",
            linewidth=1.2,
            label="fixed chance baseline",
        ),
        Patch(
            facecolor="#DCEAF3",
            edgecolor="none",
            label="fixture chance range",
        ),
    ]

    legend1 = ax.legend(
        handles=condition_handles,
        loc="lower right",
        frameon=False,
        fontsize=30,
    )
    ax.add_artist(legend1)

    model_legend = fig.legend(
        handles=model_handles,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.95),
        ncol=4,
        frameon=False,
        title="Model",
        title_fontsize=30,
        fontsize=30,
        handlelength=1.5,
        columnspacing=1.2,
        labelspacing=0.3,
    )


    # Styling.
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="x", color="#E2E2E2", linewidth=0.8)
    ax.tick_params(axis="both", which="major", length=4)

    fig.text(
        0.145,
        0.055,
        "Note: \n1. Qwen3-VL Obj-Obj Rel Pos uses old 4-option setup; not rerun due to long-video failures (Ollama Cloud will disable this model soon).\n2. Nearest Fixture chance varies by kitchen.",
        ha="left",
        va="bottom",
        fontsize=24,
        color="#444444",
    )

    fig.tight_layout(rect=[0, 0.075, 0.86, 0.95])

    png_path = OUT_DIR / "vlm_chance_aware_dotline.png"
    pdf_path = OUT_DIR / "vlm_chance_aware_dotline.pdf"

    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved dot-line plot: {png_path}")
    print(f"Saved dot-line plot: {pdf_path}")


# ============================================================
# LaTeX table generation
# ============================================================

def build_summary_table(df: pd.DataFrame) -> pd.DataFrame:
    model_order = (
        df.groupby("model")["accuracy"]
        .mean()
        .sort_values(ascending=False)
        .index
        .tolist()
    )

    rows = []

    for model in model_order:
        row = {"Model": model}

        for q in QUESTION_ORDER:
            with_video = get_score(df, model, q, VIDEO_COND)
            without_video = get_score(df, model, q, NO_VIDEO_COND)

            row[q] = f"{with_video * 100:.1f} / {without_video * 100:.1f}"

        row["Avg."] = f"{df[df['model'] == model]['accuracy'].mean() * 100:.1f}"
        row["Group"] = MODEL_GROUPS.get(model, "Models")
        rows.append(row)

    return pd.DataFrame(rows)


def latex_escape(text: str) -> str:
    return (
        str(text)
        .replace("\\", "\\textbackslash{}")
        .replace("_", "\\_")
        .replace("%", "\\%")
        .replace("&", "\\&")
        .replace("#", "\\#")
    )


def generate_latex_table(df: pd.DataFrame):
    table = build_summary_table(df)

    # Highlight best and second-best by question column using average of two conditions.
    highlight = {}

    for q in QUESTION_ORDER:
        scores = []

        for model in table["Model"]:
            subset = df[
                (df["model"] == model)
                & (df["question_class"] == q)
            ]
            mean_score = subset["accuracy"].mean()
            scores.append((model, mean_score))

        scores = sorted(scores, key=lambda x: x[1], reverse=True)

        if len(scores) >= 1:
            highlight[(scores[0][0], q)] = "best"
        if len(scores) >= 2:
            highlight[(scores[1][0], q)] = "second"

    # Highlight average column too.
    avg_scores = []

    for model in table["Model"]:
        avg = df[df["model"] == model]["accuracy"].mean()
        avg_scores.append((model, avg))

    avg_scores = sorted(avg_scores, key=lambda x: x[1], reverse=True)

    if len(avg_scores) >= 1:
        highlight[(avg_scores[0][0], "Avg.")] = "best"
    if len(avg_scores) >= 2:
        highlight[(avg_scores[1][0], "Avg.")] = "second"

    columns = [
        "Model",
        "Avg.",
        "Visibility",
        "Last visible",
        "Last placement",
        "Fixture",
        "Camera rel.",
        "Obj. distance",
        "Obj. relation",
    ]

    short_headers = {
        col: display_name(col, "table")
        for col in columns
    }

    baseline_headers = {
        "Model": "",
        "Avg.": "",
        "Visibility": "50\\%",
        "Last visible": "cont.",
        "Last placement": "cont.",
        "Fixture": "20--33\\%",
        "Camera rel.": "25\\%",
        "Obj. distance": "33\\%",
        "Obj. relation": "33\\%",
    }

    lines = []

    lines.append("% Requires:")
    lines.append("% \\usepackage{booktabs}")
    lines.append("% \\usepackage[table]{xcolor}")
    lines.append("% \\usepackage{makecell}")
    lines.append("% \\usepackage{array}")
    lines.append("")
    lines.append("\\begin{table*}[t]")
    lines.append("\\centering")
    lines.append("\\small")
    lines.append("\\setlength{\\tabcolsep}{4.2pt}")
    lines.append("\\renewcommand{\\arraystretch}{1.12}")
    lines.append("\\definecolor{SectionGreen}{RGB}{225, 250, 225}")
    lines.append("\\definecolor{BestGray}{RGB}{190, 190, 190}")
    lines.append("\\definecolor{SecondGray}{RGB}{225, 225, 225}")
    lines.append("\\begin{tabular}{l c c c c c c c c}")
    lines.append("\\toprule")

    # Top grouped header.
    lines.append(
        "\\multicolumn{1}{c}{Methods} & "
        "\\multicolumn{1}{c}{Avg.} & "
        "\\multicolumn{7}{c}{Out-of-Sight Spatial Memory Questions} \\\\"
    )
    lines.append("\\cmidrule(lr){3-9}")

    # Question headers.
    header_line = " & ".join(
        [f"\\makecell{{{latex_escape(short_headers[col])}}}" for col in columns]
    )
    lines.append(header_line + " \\\\")

    # Baseline row.
    baseline_line = " & ".join(
        [
            "\\makecell{Baseline}" if col == "Model"
            else f"\\makecell{{{baseline_headers[col]}}}"
            for col in columns
        ]
    )

    lines.append("\\rowcolor{SectionGreen}")
    lines.append(baseline_line + " \\\\")
    lines.append("\\midrule")

    # Sectioned body.
    current_group = None

    for _, row in table.iterrows():
        group = row["Group"]

        if group != current_group:
            lines.append(
                f"\\rowcolor{{SectionGreen}} "
                f"\\multicolumn{{9}}{{l}}{{\\textit{{{latex_escape(group)}}}}} \\\\"
            )
            current_group = group

        cell_values = []

        for col in columns:
            if col == "Model":
                cell_values.append(latex_escape(display_name(row[col], "table")))
                continue

            value = latex_escape(row[col])

            style = highlight.get((row["Model"], col))

            if style == "best":
                value = f"\\cellcolor{{BestGray}}\\textbf{{{value}}}"
            elif style == "second":
                value = f"\\cellcolor{{SecondGray}}{value}"

            cell_values.append(value)

        lines.append(" & ".join(cell_values) + " \\\\")

    lines.append("\\bottomrule")
    lines.append("\\end{tabular}")
    lines.append("\\caption{")
    lines.append(
        f"Out-of-sight spatial memory evaluation across VLMs. "
        f"Each task cell reports {latex_escape(VIDEO_COND)} / {latex_escape(NO_VIDEO_COND)} "
        f"accuracy in percentage points. "
        f"The baseline row reports random-guess performance where applicable. "
        f"\\texttt{{cont.}} denotes continuous or structured prediction tasks for which simple "
        f"random-guess accuracy is not meaningful. "
        f"Dark gray marks the best model per column and light gray marks the second-best model."
    )
    lines.append("}")
    lines.append("\\label{tab:oos_vlm_results}")
    lines.append("\\end{table*}")

    latex_code = "\n".join(lines)

    tex_path = OUT_DIR / "vlm_results_table.tex"

    with open(tex_path, "w", encoding="utf-8") as f:
        f.write(latex_code)

    print(f"Saved LaTeX table: {tex_path}")

    return latex_code


# ============================================================
# Main
# ============================================================

if __name__ == "__main__":
    df = load_or_generate_data(CSV_PATH)

    # Save the processed data used by the script.
    df.to_csv(OUT_DIR / "vlm_results_used.csv", index=False)

    plot_chance_aware_dotline(df)

    latex_code = generate_latex_table(df)

    print("\nLaTeX table preview:\n")
    print(latex_code)

Saved dot-line plot: vlm_visualization_outputs/vlm_chance_aware_dotline.png
Saved dot-line plot: vlm_visualization_outputs/vlm_chance_aware_dotline.pdf
Saved LaTeX table: vlm_visualization_outputs/vlm_results_table.tex

LaTeX table preview:

% Requires:
% \usepackage{booktabs}
% \usepackage[table]{xcolor}
% \usepackage{makecell}
% \usepackage{array}

\begin{table*}[t]
\centering
\small
\setlength{\tabcolsep}{4.2pt}
\renewcommand{\arraystretch}{1.12}
\definecolor{SectionGreen}{RGB}{225, 250, 225}
\definecolor{BestGray}{RGB}{190, 190, 190}
\definecolor{SecondGray}{RGB}{225, 225, 225}
\begin{tabular}{l c c c c c c c c}
\toprule
\multicolumn{1}{c}{Methods} & \multicolumn{1}{c}{Avg.} & \multicolumn{7}{c}{Out-of-Sight Spatial Memory Questions} \\
\cmidrule(lr){3-9}
\makecell{Methods} & \makecell{Avg.} & \makecell{Visibility} & \makecell{Last Vis} & \makecell{Last Place} & \makecell{Nearest Fixture} & \makecell{obj-Cam Rel Pos} & \makecell{Obj-Obj Distance} & \makecell{Obj-Obj Rel Pos} \\
\ro

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from pathlib import Path


Prediction Distribution Histograms

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import textwrap
from pathlib import Path

# Data transcribed from the attached document
models = ["Qwen3.6-27B", "Qwen3-VL-235B", "InternVL3_5-8B", "VLM-3r", "GT"]

data = {
    "Obj-Cam Rel Pos": {
        "GT": {"Back-right": 41, "Front-right": 33, "Back-left": 30, "Front-left": 16},
        "Qwen3.6-27B": {"Front-left": 61, "Front-right": 56, "Back-right": 3},
        "Qwen3-VL-235B": {"Front-left": 53, "Front-right": 49, "Back-right": 10, "Back-left": 8},
        "InternVL3_5-8B": {"Front-right": 73, "Front-left": 47},
        "VLM-3r": {"Front-left": 69, "Front-right": 45, "NaN": 6},
    },
    "Obj-Obj Rel Pos": {
        "GT": {"12 to 4:30 o'clock": 44, "4:30 to 7:30 o'clock": 46, "7:30 to 12 o'clock": 30},
        "Qwen3.6-27B": {"12 to 4:30 o'clock": 61, "4:30 to 7:30 o'clock": 12, "7:30 to 12 o'clock": 47},
        "Qwen3-VL-235B": {"12 to 4:30 o'clock": 0, "4:30 to 7:30 o'clock": 0, "7:30 to 12 o'clock": 0},
        "InternVL3_5-8B": {"4:30 to 7:30 o'clock": 117, "7:30 to 12 o'clock": 2, "12 to 4:30 o'clock": 1},
        "VLM-3r": {"12 to 4:30 o'clock": 53, "4:30 to 7:30 o'clock": 38, "7:30 to 12 o'clock": 23, "NaN": 6},
    },
    "Obj-Obj Distance": {
        "GT": {"close": 35, "medium": 44, "far": 41},
        "Qwen3.6-27B": {"close": 106, "far": 13, "medium": 1},
        "Qwen3-VL-235B": {"close": 111, "medium": 8, "far": 1},
        "InternVL3_5-8B": {"close": 87, "medium": 25, "far": 8},
        "VLM-3r": {"close": 97, "medium": 14, "NaN": 6, "far": 3},
    },
    "Nearest Fixture": {
        "GT": {
            "counter area beside the hob and near the sink": 28,
            "counter area below the boiler": 15,
            "counter area close to the microwave": 13,
            "counter area between the fridge and the hob": 13,
            "counter area next to the window": 12,
            "counter area beside the hob and close to the door": 10,
            "counter area between the hob and the sink": 10,
            "sink": 7,
            "fridgefreezer": 4,
            "hob": 3,
            "windowsill": 2,
            "shelf": 2,
            "drawer": 1,
        },
        "Qwen3.6-27B": {
            "counter area beside the hob and near the sink": 55,
            "counter area between the fridge and the hob": 15,
            "counter area beside the hob and close to the door": 15,
            "sink": 6,
            "counter area close to the microwave": 6,
            "counter area between the hob and the sink": 5,
            "counter": 5,
            "counter area below the boiler": 4,
            "windowsill": 2,
            "hob": 2,
            "fridgefreezer": 2,
            "drawer": 1,
            "cupboard": 1,
            "counter area next to the window": 1,
        },
        "Qwen3-VL-235B": {
            "counter area beside the hob and near the sink": 58,
            "counter area between the hob and the sink": 19,
            "counter area close to the microwave": 11,
            "sink": 7,
            "counter area below the boiler": 6,
            "counter area between the fridge and the hob": 5,
            "counter": 5,
            "fridgefreezer": 4,
            "counter area next to the window": 2,
            "windowsill": 1,
            "drawer": 1,
            "hob": 1,
        },
        "InternVL3_5-8B": {
            "counter area beside the hob and near the sink": 42,
            "counter area next to the window": 30,
            "counter area below the boiler": 21,
            "sink": 7,
            "counter area between the hob and the sink": 4,
            "shelf": 4,
            "counter area beside the hob and close to the door": 4,
            "counter": 3,
            "windowsill": 2,
            "hob": 2,
            "cupboard": 1,
        },
        "VLM-3r": {
            "counter area beside the hob and near the sink": 51,
            "counter area between the fridge and the hob": 14,
            "counter area between the hob and the sink": 12,
            "counter area beside the hob and close to the door": 8,
            "NaN": 6,
            "counter area close to the microwave": 5,
            "sink": 5,
            "counter area below the boiler": 4,
            "drawer": 3,
            "hob": 3,
            "fridgefreezer": 3,
            "counter area next to the window": 2,
            "counter": 2,
            "windowsill": 1,
            "shelf": 1,
        },
    }
}

# Convert to dataframes and plot
def make_df(metric):
    cats = []
    for m in models:
        cats.extend(data[metric].get(m, {}).keys())
    # Preserve GT order first, then unseen prediction-only labels
    ordered = list(data[metric]["GT"].keys())
    for c in cats:
        if c not in ordered:
            ordered.append(c)
    return pd.DataFrame(
        {m: [data[metric].get(m, {}).get(c, 0) for c in ordered] for m in models},
        index=ordered
    )

out_dir = Path("/work/courses/3dv/team1/lmms-eval/diagonistic/vlm_visualization_outputs")
png_path = out_dir / "compact_prediction_distribution_histograms.png"
pdf_path = out_dir / "compact_prediction_distribution_histograms.pdf"

fig, axes = plt.subplots(2, 2, figsize=(16, 9), constrained_layout=True)
axes = axes.flatten()

for ax, metric in zip(axes, data.keys()):
    df = make_df(metric)
    x = np.arange(len(df.index))
    width = 0.16
    
    for i, m in enumerate(models):
        ax.bar(x + (i - 2) * width, df[m].values, width, label=m)
    
    ax.set_ylabel("Count")
    ax.set_xticks(x)
    
    if metric == "Nearest Fixture":
        labels = ["\n".join(textwrap.wrap(s, 18)) for s in df.index]
        ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=21)
    else:
        labels = ["\n".join(textwrap.wrap(s, 12)) for s in df.index]
        ax.set_xticklabels(labels, rotation=0, ha="center", fontsize=24)
    
    ax.grid(axis="y", alpha=0.25)
    ax.margins(x=0.01)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=len(models), frameon=False, bbox_to_anchor=(0.5, 1.04))

fig.savefig(png_path, dpi=220, bbox_inches="tight")
fig.savefig(pdf_path, bbox_inches="tight")
plt.close(fig)